# Docker Runtime Diagnostics

Checks Docker Compose, running services, effective Redis environment, app logs, and Redis keyspace. Cells are safe to run locally; Docker failures are reported rather than hidden.

In [ ]:
from pathlib import Path
import subprocess

cwd = Path.cwd().resolve()
project_root = next((p for p in [cwd, *cwd.parents] if (p / 'compose.yml').exists()), cwd)
print('project_root:', project_root)
print('compose:', project_root / 'compose.yml')

In [ ]:
def run(cmd, timeout=30):
    print('\n$', ' '.join(cmd))
    try:
        result = subprocess.run(cmd, cwd=project_root, capture_output=True, text=True, timeout=timeout)
        print(result.stdout)
        if result.stderr:
            print('STDERR:', result.stderr)
        print('returncode:', result.returncode)
        return result
    except Exception as exc:
        print('ERROR:', repr(exc))
        return None

run(['docker', 'compose', 'ps'])

In [ ]:
cfg = run(['docker', 'compose', 'config'], timeout=60)
if cfg and cfg.returncode == 0:
    for line in cfg.stdout.splitlines():
        if 'REDIS_' in line or 'PYTHONPATH' in line:
            print(line)

In [ ]:
run([
    'docker', 'compose', 'exec', '-T', 'app', 'python', '-c',
    "import os, socket; h=os.getenv('REDIS_HOST'); print('REDIS_HOST=', h); print('REDIS_PORT=', os.getenv('REDIS_PORT')); print('REDIS_ENABLED=', os.getenv('REDIS_ENABLED')); print('REDIS_PASSWORD_SET=', bool(os.getenv('REDIS_PASSWORD'))); print('resolve=', socket.gethostbyname(h))"
])

In [ ]:
run(['docker', 'compose', 'exec', '-T', 'redis', 'redis-cli', 'ping'])
run(['docker', 'compose', 'exec', '-T', 'redis', 'redis-cli', 'dbsize'])
run(['docker', 'compose', 'exec', '-T', 'redis', 'redis-cli', '--scan'])

In [ ]:
run(['docker', 'compose', 'logs', '--tail=80', 'app'], timeout=60)

Useful commands:

```powershell
docker compose up -d --build app
docker compose logs -f app
docker compose exec redis redis-cli MONITOR
docker compose exec -T redis redis-cli --scan
```